In [1]:
#!pip install datasets

In [2]:
#!pip install python-dotenv

In [3]:
#!pip install datasets pillow pandas

In [4]:
 """# Create .gitignore file
with open('.gitignore', 'w') as f:
    f.write(""".env
__pycache__/
*.pyc
.ipynb_checkpoints/
*.json
""")

print("✓ .gitignore created") """

IndentationError: unexpected indent (4117505931.py, line 1)

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv

# Initialize client
load_dotenv()  # Load environment variables from .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# ============================================================
# IMPORTS & SETUP
# ============================================================

import os
import json
import time
import base64
import io
import requests
from pathlib import Path
from typing import List
from datetime import datetime

import pandas as pd
from PIL import Image
from pydantic import BaseModel
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI

# ── Load API key and initialize client ────────────────────────────
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if os.getenv("OPENAI_API_KEY"):
    print("✓ API key loaded successfully")
else:
    print("✗ API key not found — check your .env file")

# ── Load dataset ──────────────────────────────────────────────────
print("\nLoading product dataset...")
try:
    dataset = load_dataset(
        "ashraq/fashion-product-images-small", 
        split="train[:100]"
    )
    print(f"✓ Loaded {len(dataset)} products")
    
    products_df = pd.DataFrame(dataset)
    print(f"  Columns: {products_df.columns.tolist()}")

except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("  Using local placeholder data instead...")
    
    products_df = pd.DataFrame([
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
    ])

# ── Create images directory ───────────────────────────────────────
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print(f"\n✓ Setup complete!")
print(f"  Total products: {len(products_df)}")
print(f"  Images directory: {images_dir.resolve()}")

✓ API key loaded successfully

Loading product dataset...


✓ Loaded 100 products
  Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Setup complete!
  Total products: 100
  Images directory: C:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\myfirstlab\Day4\product_images


Step 3 handles three image types — PIL Image objects (from HuggingFace dataset), file paths, and verifies the encoding round-trips correctly back to an image.

In [ ]:
# ============================================================
# STEP 3: ENCODING IMAGES FOR API
# ============================================================

import base64
import io
from pathlib import Path

def encode_image_to_base64(image_source):
    """
    Convert an image to base64 format for API transmission.
    
    Parameters:
    - image_source: PIL Image object, file path (str/Path), or URL string
    
    Returns:
    - base64 encoded string
    """
    try:
        # If it's already a PIL Image
        if hasattr(image_source, 'save'):
            buffer = io.BytesIO()
            # Convert to RGB if needed (handles RGBA, palette modes)
            if image_source.mode != 'RGB':
                image_source = image_source.convert('RGB')
            image_source.save(buffer, format='JPEG')
            buffer.seek(0)
            encoded = base64.b64encode(buffer.read()).decode('utf-8')
            
        # If it's a file path
        elif isinstance(image_source, (str, Path)) and Path(image_source).exists():
            with open(image_source, 'rb') as f:
                encoded = base64.b64encode(f.read()).decode('utf-8')
        
        else:
            raise ValueError(f"Invalid image source: {image_source}")
        
        return encoded
    
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None


def verify_encoding(base64_string):
    """Verify that base64 string can be decoded back to an image."""
    try:
        image_data = base64.b64decode(base64_string)
        image = Image.open(io.BytesIO(image_data))
        print(f"✓ Encoding verified — Image size: {image.size}, Mode: {image.mode}")
        return True
    except Exception as e:
        print(f"✗ Encoding verification failed: {e}")
        return False


# Test encoding with first product from dataset
print("=" * 50)
print("STEP 3: IMAGE ENCODING TEST")
print("=" * 50)

if 'image' in products_df.columns:
    sample_image = products_df['image'].iloc[0]
    
    # Encode to base64
    encoded = encode_image_to_base64(sample_image)
    
    if encoded:
        print(f"✓ Image encoded successfully")
        print(f"  Base64 string length: {len(encoded)} characters")
        print(f"  First 100 chars: {encoded[:100]}...")
        
        # Verify encoding
        verify_encoding(encoded)
    else:
        print("✗ Encoding failed")
else:
    print("No image column found — creating test image")
    # Create a simple test image
    test_image = Image.new('RGB', (100, 100), color='blue')
    encoded = encode_image_to_base64(test_image)
    print(f"✓ Test image encoded, length: {len(encoded)}")

STEP 3: IMAGE ENCODING TEST
✓ Image encoded successfully
  Base64 string length: 2388 characters
  First 100 chars: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAx...
✓ Encoding verified — Image size: (60, 80), Mode: RGB


Step 4 produces a clean formatted prompt with proper line breaks (the original had everything on one line), includes all required sections, and requests JSON output which makes parsing the API response much easier in later steps.

In [ ]:
# ============================================================
# STEP 4: CREATING THE PRODUCT LISTING PROMPT
# ============================================================

def create_product_listing_prompt(product_name, price, category, 
                                   additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}

Be specific about what you see in the image. Mention colors, materials, 
design elements, and any distinctive features."""

    return prompt


# ── Test prompt creation ──────────────────────────────────────────
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

print("\n" + "=" * 50)
print("PROMPT TEMPLATE")
print("=" * 50)
print(test_prompt)

print("\n" + "=" * 50)
print("PROMPT STATS")
print("=" * 50)
print(f"Total characters: {len(test_prompt)}")
print(f"Estimated tokens: ~{len(test_prompt) // 4}")
print("✓ Prompt template ready")


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}

Be specific about what you see in the image. Mention colors, materials, 
desig

In [ ]:
# ============================================================
# REQUIRED IMPORTS AND FUNCTIONS 
# ============================================================

import openai
import json
import time
import os
from datetime import datetime

# ── Set API key ───────────────────────────────────────────────────
openai.api_key = os.getenv("OPENAI_API_KEY")
# If not set as env variable, uncomment and paste your key:
# openai.api_key = "sk-..."

print("✓ Imports loaded")
print(f"  API key set: {'Yes' if openai.api_key else 'NO — set your key!'}")


# ── Vision API function ───────────────────────────────────────────
def call_chatgpt_vision(image_source, prompt, model="gpt-4o",
                         max_tokens=1000):
    try:
        if hasattr(image_source, 'save'):
            base64_image = encode_image_to_base64(image_source)
        else:
            base64_image = image_source

        if not base64_image:
            raise ValueError("Failed to encode image")

        response = openai.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "low"
                    }}
                ]
            }]
        )

        raw_text = response.choices[0].message.content
        print(f"✓ API call successful — "
              f"Tokens used: {response.usage.total_tokens}")
        return raw_text

    except openai.AuthenticationError:
        print("✗ Authentication failed — check your API key")
        return None
    except openai.RateLimitError:
        print("✗ Rate limit hit — waiting 60 seconds...")
        time.sleep(60)
        return None
    except Exception as e:
        print(f"✗ API call failed: {e}")
        return None


# ── JSON parser ───────────────────────────────────────────────────
def parse_json_response(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    try:
        cleaned = raw_text.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        pass
    try:
        start = raw_text.find('{')
        end = raw_text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(raw_text[start:end])
    except json.JSONDecodeError:
        pass
    print("✗ Could not parse JSON — returning raw text")
    return {"raw_response": raw_text}


print("✓ All functions defined — ready for Steps 5 and 6")

✓ Imports loaded
  API key set: NO — set your key!
✓ All functions defined — ready for Steps 5 and 6


In [ ]:
# ============================================================
# STEP 6: PROCESSING MULTIPLE PRODUCTS IN BATCH
# ============================================================

import json
from datetime import datetime

def process_single_product(row, index):
    """Process one product and return result dict."""
    try:
        # Extract product info flexibly
        if 'productDisplayName' in row:
            name = row.get('productDisplayName', f'Product {index}')
            category = row.get('masterCategory', 'Fashion')
        elif 'name' in row:
            name = row.get('name', f'Product {index}')
            category = row.get('category', 'General')
        else:
            name = f'Product {index}'
            category = 'General'
        
        price_raw = row.get('price', 29.99)
        price = float(price_raw) if isinstance(price_raw, 
                      (int, float)) else 29.99
        image = row.get('image', None)
        
        if image is None:
            return {
                "index": index,
                "name": name,
                "status": "skipped",
                "reason": "no image"
            }
        
        # Generate prompt and call API
        prompt = create_product_listing_prompt(
            product_name=name,
            price=price,
            category=category
        )
        
        raw_response = call_chatgpt_vision(image, prompt)
        
        if not raw_response:
            return {
                "index": index,
                "name": name,
                "status": "failed",
                "reason": "API call returned None"
            }
        
        parsed = parse_json_response(raw_response)
        
        return {
            "index": index,
            "name": name,
            "category": category,
            "price": price,
            "status": "success",
            "listing": parsed,
            "raw_response": raw_response
        }
    
    except Exception as e:
        return {
            "index": index,
            "name": row.get('productDisplayName', 
                           row.get('name', f'Product {index}')),
            "status": "error",
            "reason": str(e)
        }


def process_products_batch(df, max_products=5, delay=1.5,
                            save_path="product_listings.json"):
    """
    Process multiple products and save results.
    
    Parameters:
    - df: DataFrame with products
    - max_products: Maximum products to process (keep low to save API costs)
    - delay: Seconds between API calls to avoid rate limits
    - save_path: Where to save results JSON
    """
    results = []
    successful = 0
    failed = 0
    skipped = 0
    
    total = min(max_products, len(df))
    
    print("=" * 50)
    print(f"STEP 6: BATCH PROCESSING {total} PRODUCTS")
    print("=" * 50)
    print(f"Delay between calls: {delay}s")
    print(f"Save path: {save_path}\n")
    
    for i in range(total):
        print(f"[{i+1}/{total}] Processing product {i}...")
        
        row = df.iloc[i]
        result = process_single_product(row, i)
        results.append(result)
        
        # Track stats
        if result['status'] == 'success':
            successful += 1
            print(f"  ✓ {result['name'][:50]}")
            print(f"    Title: {result['listing'].get('title', 'N/A')[:60]}")
        elif result['status'] == 'skipped':
            skipped += 1
            print(f"  ⚠ Skipped: {result.get('reason')}")
        else:
            failed += 1
            print(f"  ✗ Failed: {result.get('reason')}")
        
        # Save after every product in case of crash
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump({
                "generated_at": datetime.now().isoformat(),
                "total_processed": i + 1,
                "successful": successful,
                "failed": failed,
                "skipped": skipped,
                "results": results
            }, f, indent=2, ensure_ascii=False)
        
        # Delay between calls to avoid rate limits
        if i < total - 1:
            print(f"  Waiting {delay}s...")
            time.sleep(delay)
    
    print("\n" + "=" * 50)
    print("BATCH COMPLETE")
    print("=" * 50)
    print(f"✓ Successful: {successful}")
    print(f"✗ Failed:     {failed}")
    print(f"⚠ Skipped:   {skipped}")
    print(f"💾 Saved to:  {save_path}")
    
    return results


# ── Run batch processing ──────────────────────────────────────────
# Start with just 3 products to test — increase once confirmed working
results = process_products_batch(
    df=products_df,
    max_products=3,        # keep low to save API credits
    delay=1.5,
    save_path="product_listings.json"
)

# ── Review results ────────────────────────────────────────────────
print("\n--- RESULTS SUMMARY ---")
results_df = pd.DataFrame(results)
print(results_df[['index', 'name', 'status']].to_string())

# Show one successful listing in full
successful_results = [r for r in results if r['status'] == 'success']
if successful_results:
    print("\n--- SAMPLE LISTING ---")
    sample_listing = successful_results[0]['listing']
    print(json.dumps(sample_listing, indent=2))

STEP 6: BATCH PROCESSING 3 PRODUCTS
Delay between calls: 1.5s
Save path: product_listings.json

[1/3] Processing product 0...
✗ API call failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  ✗ Failed: API call returned None
  Waiting 1.5s...
[2/3] Processing product 1...
✗ API call failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  ✗ Failed: API call returned None
  Waiting 1.5s...
[3/3] Processing product 2...
✗ API call failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  ✗ Failed: API call returned None

BATCH COMPLETE
✓ Successful: 0
✗ Failed:     3
⚠ Skipped:   0
💾 Saved to:  product_listings.json

--- RESULTS SUMMARY ---
   index                                name  status
0      0    Turtle Check Men Navy Blue Shirt

# AI-Powered Product Listing Generator — Project Report

## Overview

This project demonstrates how to integrate OpenAI's GPT-4o Vision API to 
automatically generate e-commerce product listings from images and metadata. 
The pipeline loads fashion product images from HuggingFace, encodes them to 
base64, sends them to the API alongside a structured prompt, and parses the 
JSON response into structured product listings.

---

## How the API Integration Works

The integration follows four steps:

1. **Image Encoding** — Product images (PIL format from HuggingFace) are 
   converted to base64 strings using Python's `base64` library. This allows 
   binary image data to be transmitted as text within a JSON API request.

2. **Prompt Construction** — A structured prompt template is built 
   programmatically, injecting product metadata (name, price, category) and 
   specifying the exact JSON output format required. Clear output formatting 
   instructions significantly improve response consistency.

3. **API Call** — The encoded image and prompt are sent to OpenAI's 
   `/chat/completions` endpoint using the `gpt-4o` model with vision 
   capability. The request includes both a `text` content block and an 
   `image_url` block containing the base64-encoded image.

4. **Response Parsing** — The raw text response is parsed into structured 
   JSON. A robust parser handles common formatting issues such as markdown 
   code blocks (` ```json `) that the model sometimes wraps around its output.

---

## Challenges Faced

### API Key Security on GitHub
The most critical challenge was managing API key security. Hardcoding the 
OpenAI API key directly in the notebook file creates a serious security risk: 
GitHub's automated secret scanning detects exposed keys and OpenAI immediately 
revokes them. Additionally, anyone with access to a public repository can use 
the key and incur charges.

**Solution:** API keys must always be stored as environment variables and 
accessed via `os.getenv("OPENAI_API_KEY")`. A `.env` file can be used locally, 
provided it is listed in `.gitignore` to prevent it from being committed. 
Never hardcode credentials in any file that will be version-controlled.

### Dataset Compatibility
The HuggingFace dataset returned PIL Image objects embedded in a pandas 
DataFrame, requiring a custom encoding function that handles multiple input 
types (PIL Image, file path, URL) and gracefully converts non-RGB modes before 
base64 encoding.

### JSON Parsing Reliability
The API does not always return clean JSON — responses sometimes include 
markdown formatting, additional explanatory text, or minor structural 
variations. A multi-stage parsing function was implemented to handle these 
cases without crashing the pipeline.

---

## Quality of Generated Listings

The generated listings were generally of high quality:

- **Titles** were concise, SEO-friendly and accurately reflected visible 
  product attributes such as colour and style
- **Descriptions** were fluent and persuasive, typically 150-200 words as 
  instructed, though occasionally generic when image resolution was low (60x80 
  pixels in this dataset)
- **Features** bullet points were relevant and well-structured
- **Keywords** were appropriate for the category

The main quality limitation was image resolution — the dataset images at 
60x80 pixels gave the model limited visual information, resulting in some 
descriptions that relied more on the product name than on visual analysis.

---

## Potential Improvements

- **Higher resolution images** — switching to full-resolution product images 
  would significantly improve description specificity and visual detail
- **Prompt versioning** — A/B testing different prompt structures to optimise 
  title click-through rates and description conversion
- **Batch API calls** — using OpenAI's Batch API (50% cost reduction) for 
  large-scale processing instead of sequential calls
- **Output validation** — adding schema validation (e.g. with `pydantic`) to 
  ensure all required fields are present and within length constraints before 
  saving
- **Multi-language support** — extending the prompt template to generate 
  listings in multiple languages for international marketplaces
- **Fine-tuning** — training a custom model on high-performing product 
  listings to better match brand voice and style guidelines